# 02. Exploratory Data Analysis

This notebook focuses on the business-facing questions from the original draft: category mix, install behavior, pricing, release timing, and monetization signals.


In [ ]:
from pathlib import Path

HELPER_NOTEBOOK = Path("00_helpers.ipynb")
if not HELPER_NOTEBOOK.exists():
    HELPER_NOTEBOOK = Path("notebooks/00_helpers.ipynb")

HELPER_NOTEBOOK

In [ ]:
%run $HELPER_NOTEBOOK


In [ ]:
data = load_or_prepare_data()
data.head()


## Snapshot

A compact summary before looking at the charts.


In [ ]:
summary = {
    "rows": len(data),
    "categories": data["Category"].nunique(),
    "regions": data["Region"].nunique(),
    "free_share": round(data["Free"].mean(), 3),
    "median_rating": round(data["Rating"].median(), 2),
}
summary


## Category Landscape


In [ ]:
category_counts = data["Category"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(14, 7))
sns.barplot(x=category_counts.index, y=category_counts.values, palette="viridis")
plt.title("Number of Apps by Category")
plt.xlabel("Category")
plt.ylabel("App Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
mean_installs_per_category = data.groupby("Category")["Installs"].mean().sort_values(ascending=False)
plt.figure(figsize=(14, 7))
sns.barplot(x=mean_installs_per_category.index, y=mean_installs_per_category.values, palette="crest")
plt.title("Average Installs by Category")
plt.xlabel("Category")
plt.ylabel("Average Installs")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## Install Drivers


In [ ]:
sampled = sample_for_scatter(data, n=5000)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.scatterplot(data=sampled, x="Rating Confidence", y="log_installs", alpha=0.5, ax=axes[0, 0], color="#1f77b4")
axes[0, 0].set_title("Installs vs Rating Confidence")

sns.scatterplot(data=sampled, x="Size", y="log_installs", alpha=0.5, ax=axes[0, 1], color="#2ca02c")
axes[0, 1].set_title("Installs vs App Size")

sns.scatterplot(data=sampled, x="Age", y="log_installs", alpha=0.5, ax=axes[1, 0], color="#ff7f0e")
axes[1, 0].set_title("Installs vs App Age")

sns.scatterplot(data=sampled, x="Days Since Update", y="log_installs", alpha=0.5, ax=axes[1, 1], color="#d62728")
axes[1, 1].set_title("Installs vs Time Since Update")

for ax in axes.flat:
    ax.set_ylabel("log(Installs + 1)")

plt.tight_layout()
plt.show()


## Pricing and Monetization


In [ ]:
plot_data = data.copy()
plot_data["log_Installs"] = np.log1p(plot_data["Installs"])

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=plot_data,
    x="log_Installs",
    y="Price",
    hue="Free",
    palette={0: "#1f77b4", 1: "#2ca02c"},
    alpha=0.5,
    s=40,
)
plt.title("Price vs Installs")
plt.xlabel("log(Installs + 1)")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=data, x="Content Rating", y="Installs", ax=axes[0], palette="Blues_r")
axes[0].set_title("Installs by Content Rating")
axes[0].tick_params(axis="x", rotation=45)

sns.countplot(data=data, x="Free", ax=axes[1], palette="Greens")
axes[1].set_title("Free vs Paid Apps")

sns.countplot(data=data, x="Editors Choice", ax=axes[2], palette="Oranges")
axes[2].set_title("Editors' Choice Distribution")

plt.tight_layout()
plt.show()


## Market Timing


In [ ]:
yearly_installs = data.groupby("Year")["Installs"].sum().reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(data=yearly_installs, x="Year", y="Installs", palette="mako")
plt.title("Total Installs by Release Year")
plt.tight_layout()
plt.show()


In [ ]:
seasonal_leaders = data.groupby(["Category", "Season"])["Installs"].sum().reset_index()
seasonal_leaders = seasonal_leaders.loc[seasonal_leaders.groupby("Category")["Installs"].idxmax()]

plt.figure(figsize=(14, 7))
sns.barplot(data=seasonal_leaders, x="Category", y="Installs", hue="Season", palette="tab10")
plt.title("Best Performing Release Season by Category")
plt.xlabel("Category")
plt.ylabel("Total Installs")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
region_installs = data.groupby("Region")["Installs"].sum().sort_values(ascending=False).reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(data=region_installs, x="Region", y="Installs", palette="viridis")
plt.title("Total Installs by Region")
plt.xlabel("Region")
plt.ylabel("Total Installs")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Correlation View


In [ ]:
corr_columns = [
    "Installs",
    "Price",
    "Size",
    "Free",
    "Ad Supported",
    "In App Purchases",
    "Editors Choice",
    "Age",
    "Days Since Update",
    "Rating Confidence",
    "Monetization Score",
]
correlation_matrix = data[corr_columns].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
